# Admin Boundaries — Fixed-Schema Columnar Walkthrough

End-to-end demo for the recipe in [issue #447](https://github.com/un-fao/GeoID/issues/447):

`gdalinfo` → fixed schema → collection w/ sidecars → routing (PG primary · ES INDEX · DuckDB BACKUP) →
GCS upload → ingestion w/ `GcsDetailedReporter` → STAC + OGC Features search via Elasticsearch → MVT tiles.

The fixture under `fixtures/admin_boundaries.geojson` carries an `external_id = "code"`
property (ISO 3166 alpha-3) so the IdentityMatcher chain refuses duplicates at the DB level.

> **Three platform gaps are documented inline as `Gap A/B/C` cells.** They are not blocked
> by this notebook and are tracked separately in the issue body.


## Setup — environment & client

Reads `DYNASTORE_BASE_URL` + `DYNASTORE_SYSADMIN_TOKEN` from `.env` (or the
process env). When pointing at `localhost`, the GCS / Pub/Sub steps are
auto-skipped — see `_GCP_ENABLED` and `PUBSUB_ENABLED`.

In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)

RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"admin-bnd-{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "countries")
REPORT_BUCKET = os.environ.get("REPORT_BUCKET", "geoid-review-temp")

IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL
PUBSUB_ENABLED = not IS_LOCAL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"REPORT_BUCKET : gs://{REPORT_BUCKET}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
print(f"PUBSUB        : {'enabled (remote)' if PUBSUB_ENABLED else 'disabled (local)'}")


## Step 0 — Discover the fixed schema (`gdalinfo` → STAC fields)

The schema is derived once from a representative file — for vector data,
`gdal_to_stac_fields` runs on the OGR layer info and emits
`raster:bands` / per-attribute types that we lift into
`CollectionPluginConfig.sidecars[*].attribute_schema`.

> **Gap A — schema bootstrap is a manual sequence today.** No
> `bootstrap-schema` endpoint exists; productising one would *not* be an
> OGC conformance class, so the platform composes existing standards
> instead. Two equivalent paths:
>
> * **(A) Offline** — hand-roll the OGR field list (cell below) and PUT it
>   straight into the PG sidecar config. Use this when you don't have a
>   sample asset uploaded yet.
> * **(B) From an uploaded sample asset** — run the platform's `gdal`
>   Process (OGC API - Processes) against an existing asset to extract the
>   real OGR layer info, map it to `attribute_schema`, then PUT it via
>   `/configs/.../classes/ItemsPostgresqlDriverConfig` (the existing
>   PluginConfig API). The "Step 0b" cells below show the full sequence
>   using only the OGC-standard surfaces — no bespoke endpoint.
>
> Idempotence is the operator's responsibility (read effective config →
> diff → PUT only on change). Issue #473 — closed by guidance, not by a
> new endpoint.


In [ ]:
# Source: stac_extensions/01_gdal_to_stac_transformer.ipynb (lifted, abbreviated).
# For an OGR layer (vector), `ogr_info` would be produced by
# `tasks/gdal/gdalinfo_task.py`; here we hand-roll a minimal stand-in matching
# fixtures/admin_boundaries.geojson.
ogr_info = {
    "driver": "GeoJSON",
    "layers": [{
        "name": "admin_boundaries",
        "geometry_type": "Polygon",
        "feature_count": 3,
        "extent": [-8.65, 36.62, 18.52, 60.84],
        "crs": "EPSG:4326",
        "fields": [
            {"name": "code",     "type": "String",   "width": 3},
            {"name": "name",     "type": "String",   "width": 64},
            {"name": "iso2",     "type": "String",   "width": 2},
            {"name": "region",   "type": "String",   "width": 64},
            {"name": "datetime", "type": "DateTime"},
        ],
    }],
}

# Map OGR types to the platform's attribute_schema entries.
_OGR_TYPE_MAP = {
    "String": "STRING",
    "Integer": "INTEGER",
    "Integer64": "BIGINT",
    "Real": "DOUBLE",
    "DateTime": "TIMESTAMPTZ",
    "Date": "DATE",
}
ATTRIBUTE_SCHEMA = [
    {"name": f["name"], "type": _OGR_TYPE_MAP[f["type"]], "nullable": f["name"] != "code"}
    for f in ogr_info["layers"][0]["fields"]
]
print(json.dumps(ATTRIBUTE_SCHEMA, indent=2))


### Step 0b — Derive the same schema from a real uploaded asset (OGC API - Processes)

When you already have a representative asset uploaded under any catalog/
collection, the platform's standard `gdal` Process (`OGC API - Processes`)
returns the same `ogr_info`. The full sequence below uses **only existing
OGC-standard surfaces** — `/processes/{id}/execution` for extraction and
`/configs/.../plugins/{plugin_id}` (PluginConfig API; `plugin_id` is the
snake_case driver ref `items_postgresql_driver`) for the config PUT.

The PUT below targets the **collection scope**
(`/configs/catalogs/{cat}/collections/{col}/plugins/{plugin_id}`), which is
the right granularity when one schema applies to a single collection. The
same PluginConfig API also exposes a **catalog scope**
(`/configs/catalogs/{cat}/plugins/{plugin_id}`, sysadmin) for shared
schemas that should apply by default to every collection in the catalog —
swap the URL if that fits your deployment better.

This cell is illustrative — leave `BOOTSTRAP_FROM = None` to skip and fall
back to the offline `ATTRIBUTE_SCHEMA` from the previous cell.


In [ ]:
# OPTIONAL — set BOOTSTRAP_FROM to the (catalog, collection, asset_id) triple
# of an already-uploaded sample to derive ATTRIBUTE_SCHEMA from real GDAL
# output instead of hand-rolling it. Leave as `None` to skip.

# Example: BOOTSTRAP_FROM = ("some-existing-catalog", "some-existing-collection", "sample.geojson")
BOOTSTRAP_FROM = None

# PluginConfig API uses the snake_case driver ref, NOT the Pydantic class name.
# `ItemsPostgresqlDriverConfig` is registered under plugin_id `items_postgresql_driver`
# (see `modules/storage/driver_registry.py`).
PG_DRIVER_PLUGIN_ID = "items_postgresql_driver"

if BOOTSTRAP_FROM is not None:
    src_cat, src_col, src_asset = BOOTSTRAP_FROM

    # 1. Run the standard `gdal` Process synchronously against the asset
    # (OGC API - Processes; route from `modules/processes/inventory.py`).
    r = client.post(
        f"/catalogs/{src_cat}/collections/{src_col}"
        f"/assets/{src_asset}/processes/gdal/execution",
        content=json.dumps({"inputs": {}, "response": "document"}),
    )
    assert r.status_code in (200, 201), r.text[:400]
    info = r.json().get("info") or r.json()  # SYNC response shape

    # 1a. Vector-only guard. `gdalinfo` against a raster asset returns
    # `bands` instead of `layers` (see `modules/gdal/models.py` VectorInfo /
    # RasterInfo discriminator). Deriving an attribute schema from a raster
    # makes no sense, so fail loudly rather than silently PUT'ing an empty
    # schema (the previous `info.get("layers") or []` path swallowed this).
    if "layers" not in info:
        raise RuntimeError(
            f"Asset {src_asset!r} is not a vector source "
            f"(gdalinfo returned keys: {sorted(info.keys())!r}). "
            "Step 0b derives an attribute schema from OGR field types; "
            "raster assets have no per-attribute schema to bootstrap."
        )

    # 2. Map OGR field types to the platform's PostgresType enum (matches
    # `modules/storage/drivers/pg_sidecars/attributes_config.py:PostgresType`:
    # TEXT INTEGER BIGINT NUMERIC FLOAT BOOLEAN TIMESTAMPTZ DATE JSONB UUID).
    # List OGR types degrade to JSONB (lossy on element type but preserves
    # array semantics). Time-of-day has no PostgresType equivalent — kept
    # as TEXT explicitly so the fallback is visible at the call site.
    # Unknown OGR types still fall back to TEXT via `.get(..., "TEXT")`.
    _OGR_TO_PG = {
        "String": "TEXT",
        "Integer": "INTEGER",
        "Integer64": "BIGINT",
        "Real": "NUMERIC",
        "Boolean": "BOOLEAN",
        "DateTime": "TIMESTAMPTZ",
        "Date": "DATE",
        "Time": "TEXT",
        "IntegerList": "JSONB",
        "Integer64List": "JSONB",
        "RealList": "JSONB",
        "StringList": "JSONB",
    }
    layers = info["layers"] or []
    fields = (layers[0].get("fields") if layers else []) or []
    ATTRIBUTE_SCHEMA = [
        {"name": f["name"], "type": _OGR_TO_PG.get(f["type"], "TEXT"), "nullable": True}
        for f in fields
    ]
    print("Derived from", src_asset, ":")
    print(json.dumps(ATTRIBUTE_SCHEMA, indent=2))

    # 3. (Idempotent PUT pattern) Read the effective items_postgresql_driver
    # config at the *target* (CATALOG_ID, COLLECTION_ID). PUT only if the
    # attributes-sidecar schema actually changed — re-running with the same
    # asset would otherwise bump server-side revision metadata on every run.
    cfg_url = (
        f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
        f"/plugins/{PG_DRIVER_PLUGIN_ID}"
    )
    r = client.get(cfg_url)
    pg_cfg = r.json() if r.status_code == 200 else {}
    sidecars = list(pg_cfg.get("sidecars") or [])
    existing_schema = None
    found = False
    for sc in sidecars:
        if sc.get("sidecar_type") == "attributes":
            found = True
            existing_schema = sc.get("attribute_schema")
            sc["attribute_schema"] = ATTRIBUTE_SCHEMA
            break
    if not found:
        sidecars.append({"sidecar_type": "attributes", "attribute_schema": ATTRIBUTE_SCHEMA})
    pg_cfg["sidecars"] = sidecars

    if found and existing_schema == ATTRIBUTE_SCHEMA:
        print("PG driver config already up-to-date — skipping PUT.")
    else:
        r = client.put(cfg_url, content=json.dumps(pg_cfg))
        print(f"PG driver config PUT: {r.status_code}")
        assert r.status_code in (200, 204), r.text[:400]


## Step 1 — Ensure the catalog + apply catalog-scope defaults

Lifted from `_setup/00_ensure_demo_catalog.ipynb`. POST is idempotent
(201 = created, 409 = already there). Catalog defaults pin PG as the
WRITE/READ driver so a collection POST on a fresh catalog succeeds.

In [ ]:
catalog = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": "Admin boundaries (fixed-schema demo)",
    "description": "Notebook walkthrough for issue #447.",
    "stac_version": "1.0.0",
    "links": [],
}
for attempt in range(3):
    r = client.post("/stac/catalogs", content=json.dumps(catalog))
    if r.status_code in (200, 201, 409):
        print(f"catalog: {r.status_code}")
        break
    time.sleep(2 * (attempt + 1))
assert r.status_code in (200, 201, 409), r.text[:400]

# Catalog-scope defaults — PG primary for both WRITE + READ.
defaults = {"configs": {
    "ItemsPostgresqlDriverConfig": {"enabled": True, "collection_type": "VECTOR"},
    "CollectionRoutingConfig": {"enabled": True, "operations": {
        "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
        "READ":  [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
    }},
}}
r = client.put(f"/configs/catalogs/{CATALOG_ID}/bulk", content=json.dumps(defaults))
print(f"catalog defaults: {r.status_code}")


## Step 2 — Create the collection + collection-scope sidecars

Lifted from `catalog/02_create_collection_with_layerconfig.ipynb` and
`write_policy/01_external_id_deduplication.ipynb`.

The `ItemsPostgresqlDriverConfig` PUT carries the **fixed `attribute_schema`**
from step 0 plus:

- `write_policy` — `require_external_id=true`, `external_id_field="properties.code"`,
  `on_duplicate="REFUSE_RETURN"` so re-uploads of the same `code` are refused
  cleanly (the `assets_identity_uq` constraint catches duplicates at the DB
  level too).
- `geometry_policy` — `invalid_geometry_policy="attempt_fix"`,
  `target_dimension="force_2d"`, `srid_mismatch_policy="transform"`.
- `geometry_stats` — area / length / centroid / circularity / convexity /
  aspect-ratio computed on the **2-D footprint** at write time.

In [ ]:
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": "Country admin boundaries",
    "description": "Fixed-schema country polygons keyed by ISO 3166 alpha-3 (`code`).",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
print(f"collection: {r.status_code} {r.text[:200]}")
assert r.status_code in (200, 201, 409), r.text[:400]


In [ ]:
# Fetch + mutate effective config (Immutable fields must be echoed back unchanged).
r = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/ItemsPostgresqlDriverConfig"
)
assert r.status_code == 200, r.text[:300]
pg_cfg = r.json()

pg_cfg["enabled"] = True
pg_cfg["collection_type"] = "VECTOR"

# Sidecar configuration — fixed attribute schema + geometry stats.
pg_cfg["sidecars"] = [
    {
        "kind": "attributes",
        "attribute_schema": ATTRIBUTE_SCHEMA,
    },
    {
        "kind": "geometry",
        "stats_2d": [
            "area", "length", "centroid",
            "circularity", "convexity", "aspect_ratio",
        ],
    },
]

# Write-policy: external_id mandatory, refuse duplicates by `properties.code`.
pg_cfg["write_policy"] = {
    "require_external_id": True,
    "external_id_field": "properties.code",
    "on_duplicate": "REFUSE_RETURN",
    "matchers": [
        {"kind": "external_id"},
        {"kind": "content_hash"},
    ],
}

# Geometry-fix policy.
pg_cfg["geometry_policy"] = {
    "invalid_geometry_policy": "attempt_fix",
    "target_dimension": "force_2d",
    "srid_mismatch_policy": "transform",
    "target_srid": 4326,
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/ItemsPostgresqlDriverConfig",
    content=json.dumps(pg_cfg),
)
print(f"PG driver config: {r.status_code}")
assert r.status_code in (200, 204), r.text[:400]


## Step 3 — Routing: PG primary · ES INDEX · DuckDB BACKUP (parquet)

Lifted from `routing/01_pg_primary_es_index_parquet_backup.ipynb`.

PG remains the **WRITE/READ primary**. Elasticsearch is wired as an INDEX target
(search mirror), and DuckDB as a BACKUP target emitting Parquet so subsequent
analytical / OTF readers pick it up.

> **Gap B — OTF as WRITE-primary is not live-tested in review env.** Routing
> already supports `iceberg_<ref>` / `duckdb_<ref>` as the WRITE driver, but
> the live combo verified in main is `pg + es + parquet_backup`. To swap, change
> the WRITE entry below to `{"driver_id": "ItemsIcebergDriver"}` (or DuckDB)
> and run a review-env smoke test. Note: Iceberg/DuckDB compute stats on write
> and bypass PG `ANALYZE` (`tasks/ingestion/main_ingestion.py:68`). *Not run in
> this PR — left at the verified PG-primary configuration.*


In [ ]:
# Driver configs — PG already configured in step 2; add ES + DuckDB.
es_cfg = {
    "class_key": "ItemsElasticsearchObfuscatedDriverConfig",
    "target": {"index_prefix": f"admin-bnd-{RUN_ID}-"},
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/ItemsElasticsearchObfuscatedDriverConfig",
    content=json.dumps(es_cfg),
)
print(f"ES driver config: {r.status_code}")

duck_cfg = {
    "class_key": "ItemsDuckdbDriverConfig",
    "warehouse": "/tmp/duckdb-backups",
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/ItemsDuckdbDriverConfig",
    content=json.dumps(duck_cfg),
)
print(f"DuckDB driver config: {r.status_code}")

routing_cfg = {
    "class_key": "CollectionRoutingConfig",
    "operations": {
        # Swap WRITE entry to ItemsIcebergDriver / ItemsDuckdbDriver to test
        # OTF-as-primary (Gap B).
        "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal"}],
        "READ":  [{"driver_id": "ItemsPostgresqlDriver"}],
    },
    "metadata": {"operations": {
        "INDEX":  [{"driver_id": "ItemsElasticsearchObfuscatedDriver",
                    "transformed": False, "on_failure": "warn"}],
        "BACKUP": [{"driver_id": "ItemsDuckdbDriver",
                    "transformed": False, "fmt": "parquet", "on_failure": "warn"}],
    }},
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig",
    content=json.dumps(routing_cfg),
)
print(f"routing: {r.status_code}")

# Verify the resolved routing.
r = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig/effective"
)
print(json.dumps(r.json(), indent=2)[:600])


## Step 4 — GCS bucket: `init-upload` → signed PUT → asset register

Lifted from `gcp/01_bucket_init_upload_and_ingest.ipynb` and
`ui_walkthrough/02_upload_with_reporter.ipynb`. Auto-skips against
`localhost`.

In [ ]:
ASSET_ID = f"admin-bnd-{RUN_ID}"
FILENAME = f"{ASSET_ID}.geojson"
CONTENT_TYPE = "application/geo+json"

init_payload = {
    "filename": FILENAME,
    "content_type": CONTENT_TYPE,
    "asset": {
        "asset_id": ASSET_ID,
        "asset_type": "VECTORIAL",
        "metadata": {"source": "admin-bnd-walkthrough"},
    },
    "upload_options": {"predefined_acl": "publicRead", "if_generation_match": 0},
}

r = client.post(
    f"/gcp/buckets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/init-upload",
    content=json.dumps(init_payload),
)
_GCP_ENABLED = r.status_code not in (404, 501, 503)
if not _GCP_ENABLED:
    print(f"GCP module not active ({r.status_code}) — skipping upload + ingestion.")
else:
    assert r.status_code == 200, r.text[:300]
    init_resp = r.json()
    upload_uri = init_resp["upload_uri"]
    print(f"init-upload OK; upload_id={init_resp['upload_id']}")

    # PUT the fixture bytes to the signed URL.
    payload_bytes = open("fixtures/admin_boundaries.geojson", "rb").read()
    up = httpx.Client(timeout=120.0)
    resp = up.put(upload_uri, content=payload_bytes,
                  headers={"Content-Type": CONTENT_TYPE})
    assert resp.status_code in (200, 201), resp.text[:200]
    print(f"PUT {len(payload_bytes)} bytes → {resp.status_code}")
    up.close()

    # Discover the catalog bucket so we can register the asset URI.
    r = client.get(f"/gcp/buckets/catalogs/{CATALOG_ID}")
    assert r.status_code == 200, r.text[:200]
    bucket_info = r.json()
    BUCKET = bucket_info.get("bucket_id") or bucket_info.get("bucket_name")
    ASSET_URI = f"gs://{BUCKET}/{COLLECTION_ID}/{FILENAME}"
    print(f"asset URI: {ASSET_URI}")

    if PUBSUB_ENABLED:
        # OBJECT_FINALIZE pub/sub event auto-registers; poll briefly.
        asset_url = f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/{ASSET_ID}"
        for attempt in range(30):
            r = client.get(asset_url)
            if r.status_code == 200:
                print(f"asset materialized after {attempt+1} polls")
                break
            time.sleep(2)
        else:
            print("WARN: pub/sub did not fire — fallback to manual register")
            client.post(
                f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
                content=json.dumps({"asset_id": ASSET_ID, "asset_type": "ASSET",
                                    "uri": ASSET_URI, "metadata": {}}),
            )
    else:
        r = client.post(
            f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
            content=json.dumps({"asset_id": ASSET_ID, "asset_type": "ASSET",
                                "uri": ASSET_URI, "metadata": {}}),
        )
        assert r.status_code in (200, 201, 409), r.text[:200]
        print(f"manual register: {r.status_code}")


## Step 5 — Ingest with `GcsDetailedReporter` to a separate scratch bucket

Lifted from `ui_walkthrough/02_upload_with_reporter.ipynb`. The
`report_file_path` points at `REPORT_BUCKET` (default `gs://geoid-review-temp`)
so the JSONL detail lines never land in the catalog's primary bucket.

Re-running the same asset ID exercises the write-policy: every row is refused
(`status='refused'`, `matcher='external_id'`) and the reporter logs each one.

In [ ]:
if not _GCP_ENABLED:
    print("SKIP: GCP module disabled.")
else:
    REPORT_PREFIX = f"gs://{REPORT_BUCKET}/{CATALOG_ID}/{COLLECTION_ID}"

    ingest_inputs = {
        "catalog_id": CATALOG_ID,
        "collection_id": COLLECTION_ID,
        "ingestion_request": {
            "asset": {"asset_id": ASSET_ID, "uri": ASSET_URI, "metadata": {}},
            "column_mapping": {
                "external_id": "code",
                "attributes_source_type": "explicit_list",
                "attribute_mapping": [
                    {"source": "code",     "map_to": "code"},
                    {"source": "name",     "map_to": "name"},
                    {"source": "iso2",     "map_to": "iso2"},
                    {"source": "region",   "map_to": "region"},
                    {"source": "datetime", "map_to": "datetime"},
                ],
            },
            "time_validity_start_column": "datetime",
            "source_srid": 4326,
            "encoding": "utf-8",
            "read_batch_size": 1000,
            "database_batch_size": 1000,
            "reporting": {
                "GcsDetailedReporter": {
                    "report_file_path": REPORT_PREFIX + "/{task_id}-{timestamp_utc}.jsonl",
                    "output_format": "JSONL",
                    "report_content": "ALL",
                    "reported_fields": ["geoid", "asset_id", "code", "name"],
                }
            },
        },
    }

    # First run — accept all 3 features.
    r = client.post(
        f"/processes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes/ingestion/execution",
        content=json.dumps({"inputs": ingest_inputs, "outputs": {}}),
        headers={**headers, "Prefer": "wait=true"},
    )
    print(f"ingest #1: {r.status_code} — {r.text[:200]}")
    assert r.status_code in (200, 201), r.text[:300]

    # Second run on the SAME asset → write_policy refuses every duplicate `code`.
    r = client.post(
        f"/processes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes/ingestion/execution",
        content=json.dumps({"inputs": ingest_inputs, "outputs": {}}),
        headers={**headers, "Prefer": "wait=true"},
    )
    print(f"ingest #2 (dup): {r.status_code} — {r.text[:200]}")
    print(f"reports under {REPORT_PREFIX}/ — refused rows tagged matcher='external_id'")


## Step 6 — Read-side: STAC search via ES, OGC Features, MVT tile

Lifted from `ui_walkthrough/03_read_search_features_tiles.ipynb` and
`queryables/01_cql2_and_collection_search.ipynb`.

STAC `/search` is **ES-primary** by default since PR #185 — the
`geometry_simplified` hint is applied automatically on the search index.
OGC API Features (`/features/.../items`) hits PG directly, MVT tiles serve
from the PG geometries sidecar via `/tiles/.../{z}/{x}/{y}.mvt`.

> **Gap C — GCS tile cache config + observability lives in `fao-maps-titiler`.**
> Cache key shape, TTL, bucket, and the cache-hit metric are not currently
> surfaced through geoid's config API. A `TilesCachingConfig` knob on
> `fao-maps-titiler` (or equivalent) would let us verify "second hit served
> from bucket" inline. *Not implemented in this PR — cross-repo work.*


In [ ]:
# 1) STAC item search — ES primary.
r = client.post("/search", content=json.dumps({
    "collections": [COLLECTION_ID],
    "limit": 5,
}))
print(f"POST /search: {r.status_code}")
body = r.json() if r.status_code == 200 else {}
print(f"  numberMatched={body.get('numberMatched')}  numberReturned={body.get('numberReturned')}")
features = body.get("features", [])

# 2) OGC API Features — CQL2 attribute filter on `region`.
features_url = f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
r = client.get(features_url, params={
    "limit": 10,
    "filter": "region LIKE 'Southern%'",
    "filter_lang": "cql2-text",
})
print(f"GET features (CQL2): {r.status_code}")
if r.status_code == 200:
    fc = r.json()
    print(f"  filtered count: {fc.get('numberReturned')}")
    for f in fc.get("features", [])[:5]:
        print(f"    {f.get('id')}: {f.get('properties', {}).get('name')}")

# 3) MVT tile — z=4 covers most of Europe.
mvt_url = f"/tiles/catalogs/{CATALOG_ID}/tiles/WebMercatorQuad/4/8/5.mvt"
r = client.get(mvt_url, params={"collections": COLLECTION_ID})
print(f"GET tile: {r.status_code}, {len(r.content)} bytes "
      f"(content-type={r.headers.get('content-type')})")
try:
    import mapbox_vector_tile
    if r.status_code == 200 and r.content:
        for layer_name, layer in mapbox_vector_tile.decode(r.content).items():
            print(f"  layer {layer_name!r}: {len(layer.get('features', []))} features")
except Exception as e:
    print(f"  (mapbox_vector_tile not installed; skipped decode: {type(e).__name__})")


## Teardown

Hard-delete the catalog (cascades to collections, assets, sidecars, routing,
ES indices, DuckDB warehouse files).

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}")
print(f"DELETE /stac/catalogs/{CATALOG_ID} → {r.status_code}")
client.close()


## Recap — what this notebook proves and what it doesn't

**Proves end-to-end:**

- A collection with a fixed `attribute_schema` derived from `gdalinfo`
- `external_id="code"` makes duplicate uploads refusable at the write_policy
  layer (matchers chain) and at the DB layer (`assets_identity_uq`)
- Geometry-fix policy + 2-D stats sidecar are wired through the PG driver
- PG primary · ES INDEX · DuckDB BACKUP routing (verified live)
- Resumable GCS upload, async pub/sub asset registration, ingestion with
  `GcsDetailedReporter` to a separate scratch bucket
- STAC `/search` answers from ES, OGC Features answers from PG, MVT tiles
  serve from the PG geometries sidecar

**Does not prove (deferred — see issue #447):**

- **Gap A** — schema bootstrap is documented as a manual sequence (Step 0b
  cells: `/processes/gdal/execution` + `/configs/.../ItemsPostgresqlDriverConfig`
  PUT). No bespoke endpoint by design — the platform composes OGC-standard
  surfaces only. Closed by issue #473 with notebook guidance.
- **Gap B** — Iceberg / DuckDB as the **WRITE primary** (live review-env smoke)
- **Gap C** — GCS tile cache config + cache-hit observability via
  `fao-maps-titiler`
